# Backpropagation with real numbers

A lesson from the Python and AI School, by Dr. Behrooz Taheri

## Our network
A tiny network: 2 inputs, 2 hidden sigmoid neurons, 1 sigmoid output and the loss $L=\frac12(\hat{y}-y)^2$. To keep it simple there are no biases.

> [A 2-2-1 network with every forward value (inside the nodes) and the errors δ of the backward pass (green).](https://raw.githubusercontent.com/behrooztaheri/learn-python-ai/main/assets/img/lessons/math_backprop.png)

*A 2-2-1 network with every forward value (inside the nodes) and the errors δ of the backward pass (green).*

## Step 1: the forward pass
| Quantity | Formula | Calculation | Value |
|---|---|---|---|
| $z_1$ | $w_{11}x_1+w_{12}x_2$ | $0.1\times1+0.2\times2$ | 0.5 |
| $h_1$ | $\sigma(z_1)$ | | 0.622 |
| $z_2$ | $w_{21}x_1+w_{22}x_2$ | $0.3\times1+0.4\times2$ | 1.1 |
| $h_2$ | $\sigma(z_2)$ | | 0.750 |
| $z_o$ | $v_1h_1+v_2h_2$ | $0.5\times0.622+0.6\times0.750$ | 0.761 |
| $\hat{y}$ | $\sigma(z_o)$ | | 0.682 |
| $L$ | $\frac12(\hat{y}-y)^2$ | $\frac12(0.682-1)^2$ | 0.0507 |

## Step 2: the error of the output layer
Define the "error" of a neuron as $\delta=\partial L/\partial z$:

$$\delta_o=(\hat{y}-y)\cdot\hat{y}(1-\hat{y})=(-0.318)(0.682)(0.318)=-0.0691$$

## Step 3: sending the error back to the hidden layer
Each hidden neuron shares in the error in proportion to **the weight that connects it to the output**:

$$\delta_j=\delta_o\cdot v_j\cdot h_j(1-h_j)$$

| Neuron | Calculation | Value |
|---|---|---|
| $\delta_1$ | $-0.0691\times0.5\times0.622\times0.378$ | −0.0081 |
| $\delta_2$ | $-0.0691\times0.6\times0.750\times0.250$ | −0.0078 |

## Step 4: the gradient of each weight
The golden rule: **weight gradient = error of the destination neuron × output of the source neuron**.

| Weight | Formula | Gradient | New weight with $\eta=0.5$ |
|---|---|---|---|
| $v_1$ | $\delta_o h_1$ | −0.0430 | 0.5215 |
| $v_2$ | $\delta_o h_2$ | −0.0518 | 0.6259 |
| $w_{11}$ | $\delta_1 x_1$ | −0.0081 | 0.1041 |
| $w_{12}$ | $\delta_1 x_2$ | −0.0162 | 0.2081 |
| $w_{21}$ | $\delta_2 x_1$ | −0.0078 | 0.3039 |
| $w_{22}$ | $\delta_2 x_2$ | −0.0155 | 0.4078 |

Every gradient is negative, so every weight grows a little and the output moves from 0.682 toward 1. After this single step the loss drops from 0.0507 to about 0.0481.

## The general formulas in matrix form
For layer $l$ with input $a^{(l-1)}$, weights $W^{(l)}$ and activation f:

| Stage | Formula |
|---|---|
| Forward | $z^{(l)}=W^{(l)}a^{(l-1)}+b^{(l)},\quad a^{(l)}=f(z^{(l)})$ |
| Error of the last layer | $\delta^{(L)}=\nabla_a L\odot f'(z^{(L)})$ |
| Error of earlier layers | $\delta^{(l)}=\big(W^{(l+1)\top}\delta^{(l+1)}\big)\odot f'(z^{(l)})$ |
| Weight gradient | $\partial L/\partial W^{(l)}=\delta^{(l)}\,a^{(l-1)\top}$ |
| Bias gradient | $\partial L/\partial b^{(l)}=\delta^{(l)}$ |

$\odot$ is the element-wise product. Notice that the backward pass uses the **transpose** of the same weight matrix as the forward pass.

## Why gradients vanish or explode
At each layer δ is multiplied by $W^\top$ and by $f'(z)$. After L layers it has been multiplied L times:

| Situation | Result | Fix |
|---|---|---|
| Small $f'$ (sigmoid: at most 0.25) | The gradient vanishes | ReLU, skip connections (ResNet), batch normalization |
| Large weights | The gradient explodes | Gradient clipping, proper initialization |
| All initial weights zero | All neurons stay identical | Random initialization |

| Initialization | Variance of the weights | Suited to |
|---|---|---|
| Xavier / Glorot | $\dfrac{2}{n_{in}+n_{out}}$ | Sigmoid, tanh |
| He | $\dfrac{2}{n_{in}}$ | ReLU |

## Gradient checking
Backpropagation can always be checked with a numerical derivative: nudge one weight and see how much the loss changes. The code below does that and then trains the network to solve XOR.



In [ ]:
import numpy as np

sig = lambda z: 1 / (1 + np.exp(-z))

x = np.array([1.0, 2.0]); y = 1.0
W = np.array([[0.1, 0.2],        # weights into hidden neuron 1
              [0.3, 0.4]])       # weights into hidden neuron 2
v = np.array([0.5, 0.6])         # hidden -> output

def forward(W, v):
    h = sig(W @ x); y_hat = sig(v @ h)
    return h, y_hat, 0.5 * (y_hat - y) ** 2

h, y_hat, L = forward(W, v)
print("h =", h.round(4), "  y_hat =", round(y_hat, 4), "  L =", round(L, 4))

delta_o = (y_hat - y) * y_hat * (1 - y_hat)
delta_h = delta_o * v * h * (1 - h)
grad_v = delta_o * h
grad_W = np.outer(delta_h, x)
print("delta_o =", round(delta_o, 4), "  delta_h =", delta_h.round(4))
print("grad_v =", grad_v.round(4)); print("grad_W ="); print(grad_W.round(4))

# Gradient check for w11
eps = 1e-6; Wp = W.copy(); Wp[0, 0] += eps; Wm = W.copy(); Wm[0, 0] -= eps
print("numeric dL/dw11 =", round((forward(Wp, v)[2] - forward(Wm, v)[2]) / (2 * eps), 6), "  backprop =", round(grad_W[0, 0], 6))

W2, v2 = W - 0.5 * grad_W, v - 0.5 * grad_v
print("loss after one step:", round(forward(W2, v2)[2], 4))

# Train a 2-4-1 network on XOR with the matrix formulas
rng = np.random.default_rng(1)
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float); T = np.array([[0], [1], [1], [0]], dtype=float)
W1 = rng.normal(0, 1, (2, 4)); b1 = np.zeros(4); W2 = rng.normal(0, 1, (4, 1)); b2 = np.zeros(1)
for epoch in range(1, 5001):
    H = sig(X @ W1 + b1); Y = sig(H @ W2 + b2)                  # forward
    d2 = (Y - T) * Y * (1 - Y); d1 = (d2 @ W2.T) * H * (1 - H)  # backward
    W2 -= 1.0 * H.T @ d2; b2 -= 1.0 * d2.sum(0); W1 -= 1.0 * X.T @ d1; b1 -= 1.0 * d1.sum(0)
    if epoch % 1000 == 0: print(f"epoch {epoch}: loss = {0.5 * np.sum((Y - T) ** 2):.4f}")
print("XOR predictions:", Y.ravel().round(2))

### Exercise

In the same network, if $y=0$, what would $\delta_o$ and the gradient of $v_1$ be? What do the signs tell you?

In [ ]:
# Your answer here